# Lecture02 発展: 用語集SFTと検索の比較

配布の用語集から作ったQ&AをLoRAで学習し、学習前後と検索を比較します。正解データは生成時には渡しません。

比較する4条件は **学習前・学習後・検索のみ・学習前モデル＋検索（RAG）** です。35問の小規模評価なので、普遍的な優劣は結論にしません。

Google ColabでT4 GPUを選び、上から順に実行してください。依存版・モデルのrevision・評価問題を固定しています。

## 0. 環境を準備する

配布ノートブックと同じ依存版を使います。学習済みの変数を残さないよう、再実行時はセッションを再起動してください。

In [ ]:
# @title 0. 固定版ライブラリの導入
import subprocess, sys
from pathlib import Path
PINS = {"transformers":"5.18.0", "peft":"0.21.1", "trl":"1.14.2", "accelerate":"1.15.0", "datasets":"5.1.0"}
subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q",*[f"{k}=={v}" for k,v in PINS.items()]], check=True)
print("依存版:", PINS)


## 1. 実験コードを読み込む

提出リポジトリの固定コミットを使用します。CLIからの実行時は同じファイルを事前に転送しています。

In [ ]:
# @title 1. リポジトリの準備
ROOT = Path("/content/aie_l02")
SOURCE_REVISION = "8f807bebbed72243eb460d0819b8fa8615e5164c"
if not (ROOT / "scripts/glossary_experiment.py").exists():
    subprocess.run(["git","clone","https://github.com/shikibuton10x/lecture-ai-engineering-2026-submission.git",str(ROOT)],check=True)
    subprocess.run(["git","-C",str(ROOT),"fetch","origin",SOURCE_REVISION],check=True)
    subprocess.run(["git","-C",str(ROOT),"checkout",SOURCE_REVISION],check=True)
sys.path.insert(0,str(ROOT / "scripts"))
print("実験ルート:", ROOT, "| コード版:", SOURCE_REVISION)


## 2. 学習データと評価問題を固定する

配布スクリプトの116件に確認要求の手本16件を加えます。検証58件、評価35問とは質問文を分けます。同じ用語の定義は学習と評価で共有するため、これは未学習の用語そのものへの一般化を測る設計ではありません。

In [ ]:
# @title 2. データ作成と分割の確認
from prepare_glossary_experiment import prepare
manifest = prepare()


## 3. 学習前のモデルと採点方法を用意する

モデルはQwen3.5-0.8Bのテキスト部分、32bit、量子化なしです。正解は採点時だけ参照します。概念語グループの一致は簡易指標なので、最後に回答原文も読みます。

In [ ]:
# @title 3. モデル読み込み・環境記録
from glossary_experiment import Experiment, CONFIG
experiment = Experiment()
print(CONFIG)


## 4. 学習前を評価する

入力は質問と共通のシステム指示です。用語集本文は渡しません。

In [ ]:
# @title 4. 学習前の35問
before = experiment.evaluate("base_closedbook")


## 5. 用語集の検索だけで答える

質問の引用符内の用語、または用語集の名前・別名の最長一致から検索語を抽出し、配布のsearch_glossaryへ渡します。正解ラベルは使いません。用語名・別名の一致を中心とする検索のため、用語名を含まない質問で失敗する場合があります。これは実測する対象です。

In [ ]:
# @title 5. 配布の用語集検索
search = experiment.evaluate("search_only")


## 6. 学習前モデルに検索結果を渡す

同じ検索関数の結果を学習前モデルへ渡します。検索のみとの違いは、結果の解釈・文章生成をモデルが行う点です。

In [ ]:
# @title 6. RAGの35問
rag = experiment.evaluate("base_rag")


## 7. LoRAで教師あり学習する

132件を10周します。元モデルは固定し、r=16のadapterだけ更新します。学習時間、最後に表示されたloss、全学習の平均loss、終了時の検証lossを別々に記録します。

In [ ]:
# @title 7. LoRA学習と検証
training = experiment.train()


## 8. 同じ35問で学習後を評価する

学習前と質問・システム指示・生成設定・採点方法をそろえます。

In [ ]:
# @title 8. 学習後の35問
after = experiment.evaluate("sft_closedbook")


## 9. 比較結果と原文を保存する

result.jsonに設定・環境・学習ログ・比較指標を保存します。comparison.csvには全質問の回答を並べます。用語集外への応答は、一般知識として正しくても「講座の用語集の範囲だけ答える」という運用方針への適合で採点します。

In [ ]:
# @title 9. 比較表と結果の保存
result = experiment.save_results()
import pandas as pd
display(pd.DataFrame(result["summaries"]).T[["n","keyword_policy_rate","known_keyword_rate","unknown_abstention_rate","median_seconds"]])
print("保存先:", ROOT / "lectures/02/advanced")
